# Dataset description

smFISH data comes to us in the form of two xlsx sheets. 

The first sheet has expression data of the GAL1 mRNA
<br>
The second sheet has expression data of the DOA1 mRNA

Both sheets hold data for a total of four conditions:

GLU - glucose only

RAFF - raffinose only

GLUGAL - 3 h after transition from glucose to galactose

RAFFGAL - 30 min after transition from raffinose to galactose

# data conversion and cleanup

let's use pandas to deal with the datasets:

In [1]:
import pandas as pd

In [2]:
data = [
    '../data/FISH_results_DOA1mRNA_all_conditions.xlsx', 
    '../data/FISH_results_GAL1mRNA_all_conditions.xlsx'
]

In [3]:
### now, let's open the datasets and put them into one dataframe

In [4]:
dataframes = []

for mRNA, data_path in zip(['DOA1', 'GAL1'], data):
    
    #open data
    df = pd.read_excel(
        io=data_path,
        engine='openpyxl',
        skiprows=4
    )
    
    # add an identifier column that tells which mrna
    df['mrna'] = mRNA
    
    dataframes.append(df)

In [5]:
# inspection:

dataframes[0].head()

,FILE,CELL,AREA_cell,AREA_nuc,N_total,N_thres_Total,N_thres_Nuc,mrna
0,WTGLU_01_CY3_outline.txt,Cell_CP_1,1256.0,223.0,3,3,2,DOA1
1,WTGLU_01_CY3_outline.txt,Cell_CP_2,1522.5,353.0,4,4,0,DOA1
2,WTGLU_01_CY3_outline.txt,Cell_CP_3,1778.0,431.0,3,3,1,DOA1
3,WTGLU_01_CY3_outline.txt,Cell_CP_4,472.0,214.0,2,2,1,DOA1
4,WTGLU_01_CY3_outline.txt,Cell_CP_5,1016.0,213.5,2,2,1,DOA1


In [6]:
# inspection:

dataframes[1].head()

,FILE,CELL,AREA_cell,AREA_nuc,N_total,N_thres_Total,N_thres_Nuc,mrna
0,WTGLU_01_CY5_outline.txt,Cell_CP_1,1256.0,223.0,1,1,1,GAL1
1,WTGLU_01_CY5_outline.txt,Cell_CP_2,1522.5,353.0,2,2,0,GAL1
2,WTGLU_01_CY5_outline.txt,Cell_CP_3,1778.0,431.0,1,1,0,GAL1
3,WTGLU_01_CY5_outline.txt,Cell_CP_4,472.0,214.0,1,1,0,GAL1
4,WTGLU_01_CY5_outline.txt,Cell_CP_5,1016.0,213.5,0,0,0,GAL1


The 'FILE' column holds data from the original text file. The first part of the name of that file denotes the expeirmental condition.

The "CELL" column holds the ids of the cells. Because we are dealing with a merge dataset from different experiments, the cell identifiers by themselves are not unique.

"AREA_cell" means the segmented area of a given cell in pixels.

"AREA_nuc" means the segmented area of that cell's nucleus in pixels. 
**Note**: *the thresholding used to detect nuclei can fail in some cells. Then, manual readjustment of the threshold, or manual outline correction is required. In the dataset here, these cases have a nuclear area of 0.*

"N_total" is the total number of spot events in a cell

"N_thres_Total" is the number after threshold adjustment

"N_thres_Nuc"

"mrna" is the identifier column that we just added to keep track of what's what. 

In [7]:
## let's concatentate the datdaframes:
df = pd.concat(dataframes)

In [8]:
## using regular expressions (regex) to extract the condition:

df['condition'] = df['FILE'].str.extract('.*WT(\w*)_\d{2}')

# what this does:
# .*WT finds any number of characters that preced the letters WT
# WT(\w*)_\d{2} finds any number of word characters (AaBb...) that succeed the letters WT
# and that precede and underscore (_) that is followed by two digits (\d{2})
# the round parenthesis in the regex ensures that only its contents are returned.

# so, what we do is:
# find the characters that come after WT and that come before and underscore and two digits 
# and give back those characters. 

# these characters are the experimental conditions, so we can assign them. 

In [9]:
df['condition'].unique()

array(['GLU', 'GLUGAL', 'RAFF', 'RAFFGAL'], dtype=object)

# finally, let's store the data

In [10]:
df.to_pickle('smfish.pkl')